In [1]:
import sys
print(sys.executable)
print(sys.version)

/Users/maryamellathy/Desktop/FinGaurd/.venv/bin/python
3.12.4 (v3.12.4:8e8a4baf65, Jun  6 2024, 17:33:18) [Clang 13.0.0 (clang-1300.0.29.30)]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas:", pd.__version__)
print("FinGuard environment")

Matplotlib is building the font cache; this may take a moment.


Pandas: 3.0.5
FinGuard environment ready!


In [3]:
df_sample = pd.read_csv(
    "../data/raw/paysim.csv",
    nrows=100_000
)

print(df_sample.shape)
df_sample.head()

(100000, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df_sample.shape

(100000, 11)

In [5]:
df_sample.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            100000 non-null  int64  
 1   type            100000 non-null  str    
 2   amount          100000 non-null  float64
 3   nameOrig        100000 non-null  str    
 4   oldbalanceOrg   100000 non-null  float64
 5   newbalanceOrig  100000 non-null  float64
 6   nameDest        100000 non-null  str    
 7   oldbalanceDest  100000 non-null  float64
 8   newbalanceDest  100000 non-null  float64
 9   isFraud         100000 non-null  int64  
 10  isFlaggedFraud  100000 non-null  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 8.4 MB


In [7]:
df_sample.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [8]:
df_sample["isFraud"].value_counts()

isFraud
0    99884
1      116
Name: count, dtype: int64

In [9]:
fraud_counts = df_sample["isFraud"].value_counts()

fraud_percentage = (
    df_sample["isFraud"].value_counts(normalize=True) * 100
)

print("Counts:")
print(fraud_counts)

print("\nPercentages:")
print(fraud_percentage)

Counts:
isFraud
0    99884
1      116
Name: count, dtype: int64

Percentages:
isFraud
0    99.884
1     0.116
Name: proportion, dtype: float64


In [10]:
df_sample["type"].value_counts()

type
PAYMENT     39512
CASH_OUT    30718
CASH_IN     20185
TRANSFER     8597
DEBIT         988
Name: count, dtype: int64

In [11]:
pd.crosstab(
    df_sample["type"],
    df_sample["isFraud"]
)

isFraud,0,1
type,,
CASH_IN,20185,0
CASH_OUT,30659,59
DEBIT,988,0
PAYMENT,39512,0
TRANSFER,8540,57


In [12]:
fraud_by_type = (
    df_sample
    .groupby("type")["isFraud"]
    .agg(
        transactions="count",
        fraud_cases="sum",
        fraud_rate="mean"
    )
    .sort_values("fraud_rate", ascending=False)
)

fraud_by_type["fraud_rate_pct"] = (
    fraud_by_type["fraud_rate"] * 100
)

fraud_by_type

,transactions,fraud_cases,fraud_rate,fraud_rate_pct
type,,,,
TRANSFER,8597,57,0.006630,0.663022
CASH_OUT,30718,59,0.001921,0.192070
CASH_IN,20185,0,0.000000,0.000000
DEBIT,988,0,0.000000,0.000000
PAYMENT,39512,0,0.000000,0.000000


In [13]:
df_sample.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,99884.0,173174.873646,3.403085e+05,0.32,9952.8925,52759.675,211702.185,6419835.27
1,116.0,541578.424138,1.535067e+06,164.00,17246.0000,39077.815,296154.595,10000000.00


In [14]:
pd.crosstab(
    df_sample["isFlaggedFraud"],
    df_sample["isFraud"]
)

isFraud,0,1
isFlaggedFraud,,
0,99884,116
